# 04 · Métricas de desempenho
Lê as séries do 03 e calcula métricas, inferência, tabelas e figuras, uma fatia por vez. Só roda as fatias que o protocolo liberou.

- [ ] Parâmetros e fatias liberadas
- [ ] Carga, métricas e teste na série de referência
- [ ] Funções de inferência, sinal e capacidade
- [ ] Rotinas por fatia e figuras
- [ ] Treino
- [ ] Validação
- [ ] Teste (lido uma vez)
- [ ] Lado a lado e registro de testes

## 1. Parâmetros e fatias liberadas
Constantes do protocolo, os cortes das três fatias e a lista das liberadas. Só o Álvaro muda `LIBERADAS`, depois da Emenda 6.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt

PASTA = Path('../data/processed')
PASTA_TABELAS = Path('../results/tabelas')
PASTA_FIGURAS = Path('../results/figuras')
REFERENCIA = Path('../docs/referencia_serie_E.csv')
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

SEMENTE = 20261014
BLOCO = 6
SORTEIOS = 10000
CONFIANCA = 0.95
LAGS_NW = 6
CUSTO_BPS = 20.0
GRADE_BPS = [0, 5, 10, 15, 20, 30, 50]
FIGURAS_FATIAS = ['validacao', 'teste']
PARTICIPACAO = 0.10
DIAS_EXECUCAO = [1, 3]
LOTE_PADRAO = 100
MINIMO_CANDIDATAS_FM = 20
MINIMO_CANDIDATAS_QUINTIL = 25
CONFIGURACOES_EXPLORADAS = 317
RETORNO_AUSENTE = 0.0

FATIAS = {'treino': ('2020-09-30', '2021-12-31'),
          'validacao': ('2022-01-31', '2023-12-31'),
          'teste': ('2024-01-31', '2025-12-31')}
LIBERADAS = ['treino', 'validacao', 'teste']


def meses_da_fatia(nome):
    if nome not in LIBERADAS:
        raise PermissionError(f'{nome} ainda não foi liberada na Emenda 6')
    inicio, fim = FATIAS[nome]
    return pd.date_range(inicio, fim, freq='ME')


print('fatias liberadas:', LIBERADAS, '| bloqueadas:', [n for n in FATIAS if n not in LIBERADAS])

## 2. Carga dos dados e séries
Lê o `backtest.pkl` do 03 e as tabelas dos outros notebooks, confere que têm as mesmas datas e monta as séries brutas de custo para o período inteiro. As fatias são recortadas depois, nas células de cada uma.

In [ ]:
NOMES = ['retorno_mensal', 'elegivel', 'aprovada', 'sinal_residual', 'sinal_bruto', 'saidas', 'cdi_mensal', 'ibov_mensal',
         'liquidez_12m', 'preco_sem_ajuste_mensal', 'fatores_ff3', 'cadastro_universo', 'di_mensal', 'ipca_mensal']
dados = {n: pd.read_pickle(PASTA / f'{n}.pkl') for n in NOMES}
bt = pd.read_pickle(PASTA / 'backtest.pkl')

retorno = dados['retorno_mensal']
datas = retorno.index
elegivel, aprovada = dados['elegivel'], dados['aprovada']
sinal_residual = dados['sinal_residual']
sinal_bruto = dados['sinal_bruto'].where(sinal_residual.notna())
ultimo_com_preco = dados['saidas']['ultimo_mes_com_preco']
CORTE_SAIDA = bt['parametros']['corte_saida']

VERSOES = {'iMOM-BR': 'E_residual_filtro', 'Residual sem filtro': 'D_residual', 'Bruto com filtro': 'C_bruto_filtro', 'Bruto sem filtro': 'B_bruto', 'Só filtro': 'A_filtro'}
bruto = pd.DataFrame({k: bt['series'][v]['retorno_bruto'] for k, v in VERSOES.items()})
giro = pd.DataFrame({k: bt['series'][v]['giro'] for k, v in VERSOES.items()})
refs = bt['referencias']
cdi, ibov = refs['CDI'], refs['Ibovespa']
ipca = dados['ipca_mensal'].reindex(bruto.index)

assert bruto.index.equals(refs.index) and giro.index.equals(bruto.index)
assert dados['sinal_residual'].index.equals(datas) and dados['ipca_mensal'].index.equals(datas)
assert all(meses_da_fatia_nome in bruto.index for meses_da_fatia_nome in pd.date_range('2020-09-30', '2025-12-31', freq='ME'))


def liquido(k, bps):
    return bruto[k] - bps / 10000 * giro[k]


RET_TODO = pd.DataFrame({**{k: bruto[k] for k in VERSOES},
                         'Ibovespa': ibov, 'Universo_igual': refs['Universo_igual'], 'CDI': cdi})
print(bruto.index[0].date(), 'a', bruto.index[-1].date(), '|', len(bruto), 'meses no 03')
print('versões:', list(bruto.columns), '| referências:', list(refs.columns), '| sensibilidades:', list(bt['sensibilidades']))
print('legenda: iMOM-BR = estratégia final (residual + filtro, 36/36, ROIC acima do CDI composto de 12 meses + 5 p.p.) | Residual sem filtro | Bruto com filtro | Bruto sem filtro | Só filtro')
print('(a antiga série E, com CDI + 5 p.p. e 30/36, não entra em nenhuma tabela nem figura)')
print('retornos brutos de custo em todas as séries; o efeito do custo (0 a 50 bps por lado) está na tabela de custos')

## 3. Métricas e teste na série de referência
Funções de retorno, volatilidade, Sharpe, Sortino, queda máxima e complementares. Conferidas contra a série de referência antiga (16,17% a.a. e Sharpe 0,3666).

In [ ]:
def acumulado(r):
    return (1 + r).prod() - 1


def cagr(r):
    return (1 + r).prod() ** (12 / len(r)) - 1


def volatilidade(r):
    return r.std(ddof=1) * np.sqrt(12)


def sharpe(r, cdi):
    x = r - cdi
    return x.mean() / x.std(ddof=1) * np.sqrt(12)


def sortino(r, cdi):
    x = r - cdi
    return x.mean() / np.sqrt((np.minimum(x, 0) ** 2).mean()) * np.sqrt(12)


def curva_drawdown(r):
    c = (1 + r).cumprod()
    return c / np.maximum(c.cummax(), 1) - 1


def meses_no_fundo(r):
    abaixo = (curva_drawdown(r) < 0).astype(int)
    maior = atual = 0
    for v in abaixo:
        atual = atual + 1 if v else 0
        maior = max(maior, atual)
    return maior


def resumo(r, cdi, ibov, ipca):
    dd = curva_drawdown(r)
    dif = r - ibov
    sobe, cai = ibov > 0, ibov < 0
    q05 = r.quantile(0.05)
    return {'meses': len(r), 'retorno_acumulado': acumulado(r), 'retorno_anual': cagr(r), 'volatilidade_anual': volatilidade(r),
            'sharpe': sharpe(r, cdi), 'sortino': sortino(r, cdi), 'queda_maxima': dd.min(), 'meses_no_fundo': meses_no_fundo(r),
            'calmar': cagr(r) / abs(dd.min()) if dd.min() < 0 else np.nan,
            'melhor_mes': r.max(), 'pior_mes': r.min(), 'meses_positivos': (r > 0).mean(),
            'assimetria': stats.skew(r), 'curtose_excesso': stats.kurtosis(r),
            'VaR95_mensal': q05, 'CVaR95_mensal': r[r <= q05].mean(),
            'retorno_anual_real': ((1 + r) / (1 + ipca)).prod() ** (12 / len(r)) - 1,
            'tracking_error': dif.std(ddof=1) * np.sqrt(12), 'information_ratio': dif.mean() * 12 / (dif.std(ddof=1) * np.sqrt(12)),
            'meses_acima_ibov': (dif > 0).mean(),
            'captura_alta': r[sobe].mean() / ibov[sobe].mean() if sobe.any() else np.nan,
            'captura_baixa': r[cai].mean() / ibov[cai].mean() if cai.any() else np.nan}


referencia = pd.read_csv(REFERENCIA, parse_dates=['mes']).set_index('mes')
liq_ref = referencia['retorno_liquido']
print('conferência das fórmulas na série de referência antiga (filtro CDI + 5 p.p., histórico de 30 meses, líquida de 20 bps; só serve para conferir as fórmulas):')
print('retorno anual', round(cagr(liq_ref), 4), '| Sharpe', round(sharpe(liq_ref, referencia['cdi']), 4),
      '| queda máxima', round(curva_drawdown(liq_ref).min(), 4))
assert len(liq_ref) == 64
assert round(cagr(liq_ref), 4) == 0.1617 and round(sharpe(liq_ref, referencia['cdi']), 4) == 0.3666

## 4. Funções de inferência
Bootstrap circular em blocos de 6 meses (10.000 sorteios, semente 20261014, intervalo percentil de 95%), regressão com erros de Newey-West de 6 defasagens, Sharpe probabilístico e deflacionado.

In [ ]:
def bootstrap_diferenca_sharpe(a, b, cdi, sorteios=SORTEIOS, bloco=BLOCO, semente=SEMENTE, confianca=CONFIANCA):
    xa, xb = (a - cdi).to_numpy(), (b - cdi).to_numpy()
    n = len(xa)
    gerador = np.random.default_rng(semente)
    inicios = gerador.integers(0, n, size=(sorteios, int(np.ceil(n / bloco))))
    posicoes = ((inicios[:, :, None] + np.arange(bloco)) % n).reshape(sorteios, -1)[:, :n]
    sh = lambda m: m.mean(axis=1) / m.std(axis=1, ddof=1) * np.sqrt(12)
    d = sh(xa[posicoes]) - sh(xb[posicoes])
    observado = sharpe(a, cdi) - sharpe(b, cdi)
    cauda = (1 - confianca) / 2
    inferior, superior = np.quantile(d, [cauda, 1 - cauda])
    leitura = 'a favor' if inferior > 0 else 'contra' if superior < 0 else 'inconclusivo'
    return {'diferenca_observada': observado, 'limite_inferior': inferior, 'limite_superior': superior,
            'proporcao_menor_ou_igual_a_zero': (d <= 0).mean(), 'leitura': leitura}


def bootstrap_em_laco(a, b, cdi, sorteios, bloco=BLOCO, semente=SEMENTE):
    gerador = np.random.default_rng(semente)
    xa, xb = (a - cdi).to_numpy(), (b - cdi).to_numpy()
    n = len(xa)
    saida = np.empty(sorteios)
    for i in range(sorteios):
        inicios = gerador.integers(0, n, size=int(np.ceil(n / bloco)))
        pos = np.concatenate([(k + np.arange(bloco)) % n for k in inicios])[:n]
        saida[i] = xa[pos].mean() / xa[pos].std(ddof=1) * np.sqrt(12) - xb[pos].mean() / xb[pos].std(ddof=1) * np.sqrt(12)
    return saida


_laco = bootstrap_em_laco(liq_ref, referencia['retorno_bruto'], referencia['cdi'], 300)
_vet = bootstrap_diferenca_sharpe(liq_ref, referencia['retorno_bruto'], referencia['cdi'], sorteios=300)
assert np.isclose(np.quantile(_laco, 0.025), _vet['limite_inferior']) and np.isclose(np.quantile(_laco, 0.975), _vet['limite_superior'])
print('bootstrap vetorizado igual ao laço do código antigo: ok')


def regressao_nw(y, X, lags=LAGS_NW):
    m = sm.OLS(y, sm.add_constant(X), missing='drop').fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return m, m.conf_int(alpha=1 - CONFIANCA)


def media_e_t_nw(serie):
    m = sm.OLS(serie.to_numpy(), np.ones(len(serie))).fit(cov_type='HAC', cov_kwds={'maxlags': LAGS_NW})
    return float(m.params[0]), float(m.tvalues[0])


def sharpe_probabilistico(r, cdi, referencia=0.0):
    x = (r - cdi).to_numpy()
    sr = x.mean() / x.std(ddof=1)
    g3, g4 = stats.skew(x), stats.kurtosis(x, fisher=False)
    return stats.norm.cdf((sr - referencia) * np.sqrt(len(x) - 1) / np.sqrt(1 - g3 * sr + (g4 - 1) / 4 * sr ** 2))


def sharpe_deflacionado(r, cdi, numero_de_testes, variancia_entre_sharpes_mensais):
    x = (r - cdi).to_numpy()
    sr = x.mean() / x.std(ddof=1)
    g3, g4 = stats.skew(x), stats.kurtosis(x, fisher=False)
    euler = 0.5772156649
    esperado = np.sqrt(variancia_entre_sharpes_mensais) * ((1 - euler) * stats.norm.ppf(1 - 1 / numero_de_testes)
                                                          + euler * stats.norm.ppf(1 - 1 / (numero_de_testes * np.e)))
    estatistica = (sr - esperado) * np.sqrt(len(x) - 1) / np.sqrt(1 - g3 * sr + (g4 - 1) / 4 * sr ** 2)
    return stats.norm.cdf(estatistica), esperado * np.sqrt(12)

## 5. Funções do sinal e da capacidade
Retorno do mês seguinte com a regra de saída do 03, Fama-MacBeth, retorno por quintil, capacidade e capital mínimo.

In [ ]:
def retorno_no_mes(codigos, s, corte=CORTE_SAIDA):
    anterior = datas[datas.get_loc(s) - 1]
    valores = retorno.loc[s, codigos].copy()
    faltou = valores.isna()
    saiu = faltou & (ultimo_com_preco.reindex(codigos) == anterior).to_numpy()
    valores[saiu] = corte
    valores[faltou & ~saiu] = RETORNO_AUSENTE
    return valores


def posto_centrado(valores):
    return valores.rank() / (len(valores) + 1) - 0.5


def candidatas(t, sinais, so_aprovadas):
    ok = elegivel.loc[t].copy()
    for sinal in sinais:
        ok &= sinal.loc[t].notna()
    if so_aprovadas:
        ok &= aprovada.loc[t]
    return retorno.columns[ok.to_numpy()]


def fama_macbeth(sinais, so_aprovadas=False):
    linhas, tamanhos = {}, {}
    for t in DECISAO:
        s = datas[datas.get_loc(t) + 1]
        codigos = candidatas(t, sinais, so_aprovadas)
        if len(codigos) < MINIMO_CANDIDATAS_FM:
            continue
        y = retorno_no_mes(codigos, s)
        X = np.column_stack([np.ones(len(codigos))] + [posto_centrado(sn.loc[t, codigos]).to_numpy() for sn in sinais])
        linhas[s] = np.linalg.lstsq(X, y.to_numpy(), rcond=None)[0][1:]
        tamanhos[s] = len(codigos)
    return pd.DataFrame(linhas).T, pd.Series(tamanhos)


def retorno_por_quintil(sinal, so_aprovadas):
    linhas = {}
    for t in DECISAO:
        s = datas[datas.get_loc(t) + 1]
        codigos = candidatas(t, [sinal], so_aprovadas)
        if len(codigos) < MINIMO_CANDIDATAS_QUINTIL:
            continue
        y = retorno_no_mes(codigos, s)
        posicao = sinal.loc[t, codigos].rank(method='first')
        quintil = np.ceil(posicao / len(codigos) * 5).astype(int)
        linhas[s] = {f'Q{q}': y[quintil[quintil == q].index].mean() for q in range(1, 6)} | {'candidatas': len(codigos)}
    return pd.DataFrame(linhas).T


def capacidade(pesos, dias):
    linhas = {}
    for s in pesos.index:
        t = datas[datas.get_loc(s) - 1]
        codigos = pesos.columns[(pesos.loc[s] > 0).to_numpy()]
        vols = dados['liquidez_12m'].loc[t, codigos]
        linhas[s] = {'n': len(codigos), 'v_min': vols.min(), 'limitante': vols.idxmin(),
                     'capacidade_milhoes': len(codigos) * PARTICIPACAO * dias * vols.min() / 1000}
    return pd.DataFrame(linhas).T


def capital_minimo(pesos):
    linhas = {}
    for s in pesos.index:
        t = datas[datas.get_loc(s) - 1]
        codigos = pesos.columns[(pesos.loc[s] > 0).to_numpy()]
        precos = dados['preco_sem_ajuste_mensal'].loc[t, codigos]
        linhas[s] = {'n': len(codigos), 'preco_maximo': precos.max(), 'limitante': precos.idxmax(),
                     'capital_1_acao': len(codigos) * precos.max(), 'capital_lote_100': len(codigos) * LOTE_PADRAO * precos.max()}
    return pd.DataFrame(linhas).T


def resumo_capital(tabela):
    linhas = {}
    for medida in ['capital_1_acao', 'capital_lote_100']:
        v = tabela[medida].astype(float)
        linhas[medida] = {'primeiro_mes': v.iloc[0], 'media': v.mean(), 'mediana': v.median(), 'minimo': v.min(), 'maximo': v.max(),
                          'custo_primeiro_mes': v.iloc[0] * CUSTO_BPS / 10000}
    return pd.DataFrame(linhas).T


def aporte_inicial(tabela, r):
    valor = (1 + r.reindex(tabela.index).fillna(0.0)).cumprod().shift(1).fillna(1.0)
    linhas = {}
    for medida in ['capital_1_acao', 'capital_lote_100']:
        necessario = tabela[medida].astype(float) / valor
        linhas[medida] = {'primeiro_mes': float(tabela[medida].iloc[0]), 'mes_mais_caro': float(tabela[medida].max()),
                          'aporte_inicial_que_sustenta': necessario.max(), 'mes_que_define': necessario.idxmax().date()}
    return pd.DataFrame(linhas).T


def custo_de_equilibrio(nome):
    def margem(bps):
        return acumulado(liquido(nome, bps).loc[MESES]) - acumulado(IBOV_A)
    if margem(0.0) <= 0:
        return 0.0
    if margem(500.0) > 0:
        return np.inf
    baixo, alto = 0.0, 500.0
    for _ in range(60):
        meio = (baixo + alto) / 2
        baixo, alto = (meio, alto) if margem(meio) > 0 else (baixo, meio)
    return (baixo + alto) / 2

## 6.1 Preparar a fatia
Recorta tudo nos meses da fatia pedida e zera o registro dela. Todas as seções seguintes leem só esses meses.

In [ ]:
registro_testes = []
R = {}


def registrar(nome, comparacao, metrica):
    registro_testes.append({'fatia': FATIA, 'teste': nome, 'comparacao': comparacao, 'metrica': metrica})


def salvar_tabela(tabela, nome, indice=True):
    tabela.to_csv(PASTA_TABELAS / f'04_{FATIA}_{nome}.csv', index=indice)


def definir_fatia(nome):
    global FATIA, MESES, RET, CDI_A, IBOV_A, IPCA_A, G, PESOS_E, DECISAO
    MESES = meses_da_fatia(nome)
    FATIA = nome
    registro_testes[:] = [t for t in registro_testes if t['fatia'] != nome]
    RET = RET_TODO.loc[MESES]
    CDI_A, IBOV_A, IPCA_A = cdi.loc[MESES], ibov.loc[MESES], ipca.loc[MESES]
    G = giro.loc[MESES]
    PESOS_E = bt['pesos']['E_residual_filtro'].loc[MESES]
    DECISAO = [t for t in datas[sinal_residual.notna().any(axis=1).to_numpy()] if datas[datas.get_loc(t) + 1] in MESES]
    R.clear()
    print(f'=== fatia {nome}: {len(MESES)} meses de {MESES[0].date()} a {MESES[-1].date()} | {len(DECISAO)} datas de decisão ===')

## 6.2 Resumo e custos
Métricas de cada série (iMOM-BR, as quatro versões mais simples, Ibovespa, universo igual e CDI) e a tabela de custos de 0 a 50 bps por lado, com o custo de equilíbrio contra o Ibovespa.

In [ ]:
def secao_resumo():
    with np.errstate(invalid='ignore', divide='ignore'):
        R['resumo'] = pd.DataFrame({c: resumo(RET[c], CDI_A, IBOV_A, IPCA_A) for c in RET.columns}).T
    principais = ['meses', 'retorno_acumulado', 'retorno_anual', 'volatilidade_anual', 'sharpe', 'sortino', 'queda_maxima', 'calmar']
    complementares = ['tracking_error', 'information_ratio', 'meses_acima_ibov', 'captura_alta', 'captura_baixa', 'melhor_mes', 'pior_mes',
                      'meses_positivos', 'assimetria', 'curtose_excesso', 'VaR95_mensal', 'CVaR95_mensal', 'meses_no_fundo', 'retorno_anual_real']
    print(R['resumo'][principais].round(4).to_string())
    print(R['resumo'][complementares].round(4).to_string())
    salvar_tabela(R['resumo'], 'resumo')


def secao_custos():
    linhas = {}
    for nome in VERSOES:
        equilibrio = custo_de_equilibrio(nome)
        for bps in GRADE_BPS:
            r = liquido(nome, bps).loc[MESES]
            linhas[(nome, bps)] = {'retorno_anual': cagr(r), 'volatilidade_anual': volatilidade(r), 'sharpe': sharpe(r, CDI_A), 'sortino': sortino(r, CDI_A),
                                   'queda_maxima': curva_drawdown(r).min(), 'acima_do_ibovespa': acumulado(r) > acumulado(IBOV_A),
                                   'custo_de_equilibrio_bps': equilibrio}
    R['custos'] = pd.DataFrame(linhas).T.rename_axis(['serie', 'bps_por_lado'])
    print(R['custos'].round(4).to_string())
    print('custo de equilíbrio: custo por lado a partir do qual a série deixa de superar o Ibovespa em retorno acumulado (inf: não deixa até 500 bps)')
    print('giro bilateral médio por série:', G[list(VERSOES)].mean().round(3).to_dict())
    print('custo médio do iMOM-BR a', CUSTO_BPS, 'bps por lado (só um exemplo de leitura):', round(CUSTO_BPS / 10000 * G['iMOM-BR'].mean() * 12 * 100, 2), '% do capital por ano')
    salvar_tabela(R['custos'], 'custos')

## 6.3 Comparações com intervalo: bootstrap e alfa
Seis comparações de Sharpe (iMOM-BR contra Ibovespa, universo igual, Só filtro, Bruto sem filtro, Bruto com filtro e Residual sem filtro) e o alfa de Newey-West em cinco pares, além da exposição a MKT, SMB e HML.

In [ ]:
def secao_bootstrap():
    comparacoes = [('iMOM-BR', 'Ibovespa', 'principal'), ('iMOM-BR', 'Universo_igual', 'benchmark'), ('iMOM-BR', 'Só filtro', 'versão mais simples'),
                   ('iMOM-BR', 'Bruto sem filtro', 'versão mais simples'), ('iMOM-BR', 'Bruto com filtro', 'versão mais simples'), ('iMOM-BR', 'Residual sem filtro', 'versão mais simples')]
    linhas = {}
    for esquerda, direita, papel in comparacoes:
        r = bootstrap_diferenca_sharpe(RET[esquerda], RET[direita], CDI_A)
        r['papel'] = papel
        r['acumulado_F_menos_outro'] = acumulado(RET[esquerda]) - acumulado(RET[direita])
        linhas[f'{esquerda} contra {direita}'] = r
        registrar('bootstrap_sharpe', f'{esquerda} contra {direita}', 'diferença de Sharpe')
    R['bootstrap'] = pd.DataFrame(linhas).T
    print(R['bootstrap'].round(4).to_string())
    print('iMOM-BR contra o CDI não tem diferença de Sharpe (o excesso do CDI sobre ele mesmo é zero): ver retorno acumulado e alfa')
    salvar_tabela(R['bootstrap'], 'bootstrap_sharpe')


def secao_alfa():
    pares = [('iMOM-BR', 'Bruto sem filtro', 'residual com filtro contra bruto sem filtro'), ('Residual sem filtro', 'Bruto sem filtro', 'residual contra bruto (sem filtro nos dois)'),
             ('iMOM-BR', 'Bruto com filtro', 'residual com filtro contra bruto com filtro'), ('iMOM-BR', 'Ibovespa', 'iMOM-BR contra Ibovespa'),
             ('iMOM-BR', 'Universo_igual', 'iMOM-BR contra universo igual')]
    linhas = {}
    for e, r, descricao in pares:
        m, ic = regressao_nw(RET[e] - CDI_A, (RET[r] - CDI_A).rename('referencia'))
        a = m.params.iloc[0]
        linhas[descricao] = {'alfa_mensal': a, 'alfa_anual': (1 + a) ** 12 - 1, 'erro_padrao': m.bse.iloc[0], 't': m.tvalues.iloc[0],
                             'p': m.pvalues.iloc[0], 'ic_inferior_mensal': ic.iloc[0, 0], 'ic_superior_mensal': ic.iloc[0, 1],
                             'beta': m.params.iloc[1], 'R2': m.rsquared, 'meses': int(m.nobs)}
        registrar('alfa_newey_west', f'{e} contra {r}', 'alfa mensal')
    R['alfa'] = pd.DataFrame(linhas).T
    print(R['alfa'].round(4).to_string())
    salvar_tabela(R['alfa'], 'alfa_newey_west')
    fatores = dados['fatores_ff3'].reindex(MESES)
    linhas = {}
    for c in ['Só filtro', 'Bruto sem filtro', 'Bruto com filtro', 'Residual sem filtro', 'iMOM-BR', 'Ibovespa']:
        m, ic = regressao_nw(RET[c] - CDI_A, fatores)
        linhas[c] = {'alfa_mensal': m.params['const'], 't_alfa': m.tvalues['const'], 'beta_MKT': m.params['MKT'], 'beta_SMB': m.params['SMB'],
                     'beta_HML': m.params['HML'], 'SMB_ic_inf': ic.loc['SMB', 0], 'SMB_ic_sup': ic.loc['SMB', 1],
                     'HML_ic_inf': ic.loc['HML', 0], 'HML_ic_sup': ic.loc['HML', 1], 'R2': m.rsquared}
    R['fatores'] = pd.DataFrame(linhas).T
    print(R['fatores'].round(3).to_string())
    salvar_tabela(R['fatores'], 'exposicao_aos_fatores')

## 6.4 Poder do sinal: Fama-MacBeth e quintis
Regressão mensal do retorno do mês seguinte no posto do sinal e retorno médio por quintil, nas elegíveis e nas aprovadas.

In [ ]:
def secao_sinal():
    especificacoes = {'residual': [sinal_residual], 'bruto': [sinal_bruto], 'residual e bruto juntos': [sinal_residual, sinal_bruto]}
    linhas = []
    for amostra, so_aprovadas in [('elegíveis com sinal', False), ('aprovadas', True)]:
        for nome, sinais in especificacoes.items():
            coef, tam = fama_macbeth(sinais, so_aprovadas)
            for i, sinal_nome in enumerate(['residual', 'bruto'] if len(sinais) == 2 else [nome]):
                media, t_stat = media_e_t_nw(coef[i])
                linhas.append({'amostra': amostra, 'especificação': nome, 'sinal': sinal_nome, 'coef_mensal': media, 't_newey_west': t_stat,
                               'meses': len(coef), 'candidatas_médias': tam.mean()})
            registrar('fama_macbeth', f'{nome} | {amostra}', 'coeficiente do posto do sinal')
    R['fm'] = pd.DataFrame(linhas)
    print(R['fm'].round(4).to_string(index=False))
    print('o coeficiente é a diferença de retorno esperado no mês entre o topo e o fundo do ranking (posto centrado vai de -0,5 a +0,5)')
    salvar_tabela(R['fm'], 'fama_macbeth', indice=False)
    R['quintis'], linhas = {}, []
    for amostra, so_aprovadas in [('elegíveis com sinal', False), ('aprovadas', True)]:
        for nome, sinal in [('residual', sinal_residual), ('bruto', sinal_bruto)]:
            q = retorno_por_quintil(sinal, so_aprovadas)
            R['quintis'][(amostra, nome)] = q
            medias = q[[f'Q{i}' for i in range(1, 6)]].mean()
            media, t_stat = media_e_t_nw(q['Q5'] - q['Q1'])
            linhas.append({'amostra': amostra, 'sinal': nome, **medias.to_dict(), 'Q5_menos_Q1': media, 't_newey_west': t_stat,
                           'monotonicidade_spearman': stats.spearmanr(range(1, 6), medias.to_numpy())[0], 'meses': len(q),
                           'candidatas_médias': q['candidatas'].mean()})
            registrar('quintis', f'{nome} | {amostra}', 'Q5 menos Q1')
    R['tabela_quintis'] = pd.DataFrame(linhas)
    print(R['tabela_quintis'].round(4).to_string(index=False))
    salvar_tabela(R['tabela_quintis'], 'quintis_do_sinal', indice=False)

## 6.5 Por ano, Sharpe deflacionado, capacidade e capital mínimo
Retorno por ano civil, Sharpe probabilístico e deflacionado (12 e 317 testes) e capacidade em R$ com 10% do volume. O capital mínimo é o valor para comprar todas as ações em pesos iguais: N vezes o preço da ação mais cara, com 1 ação (mercado fracionário) ou com o lote padrão de 100 ações, no preço de fechamento sem ajuste da data de decisão.

In [ ]:
def secao_anos():
    anos = {}
    for ano, r in RET[['iMOM-BR', 'Ibovespa', 'CDI']].groupby(RET.index.year):
        anos[ano] = {'meses': len(r), 'iMOM-BR': acumulado(r['iMOM-BR']), 'Ibovespa': acumulado(r['Ibovespa']), 'CDI': acumulado(r['CDI']),
                     'vol_iMOM_mensal': r['iMOM-BR'].std(ddof=1) if len(r) > 1 else np.nan, 'queda_maxima_iMOM': curva_drawdown(r['iMOM-BR']).min()}
    R['anual'] = pd.DataFrame(anos).T
    print(R['anual'].round(4).to_string())
    print('ano com menos de 12 meses não é anualizado: a coluna vol_iMOM_mensal é o desvio mensal')
    salvar_tabela(R['anual'], 'por_ano')


def secao_deflacionado():
    todas = {k: bruto[k] for k in VERSOES}
    for nome, (tabela, _) in bt['sensibilidades'].items():
        todas[nome] = tabela['retorno_bruto']
    excessos = {n: (r.reindex(MESES) - CDI_A).dropna() for n, r in todas.items()}
    variancia = pd.Series({n: x.mean() / x.std(ddof=1) for n, x in excessos.items()}).var(ddof=1)
    linhas = []
    for n in [len(todas), CONFIGURACOES_EXPLORADAS]:
        dsr, esperado = sharpe_deflacionado(RET['iMOM-BR'], CDI_A, n, variancia)
        linhas.append({'testes': n, 'PSR_contra_zero': sharpe_probabilistico(RET['iMOM-BR'], CDI_A), 'DSR': dsr, 'Sharpe_esperado_do_melhor_ao_acaso_anual': esperado})
    R['dsr'] = pd.DataFrame(linhas)
    print(len(todas), 'variantes do 03 usadas só para estimar a variância entre Sharpes mensais:', round(variancia, 5))
    print(R['dsr'].round(4).to_string(index=False))
    print('aproximação: as 315 configurações da exploração não foram guardadas, então a variância vem das variantes do 03; declarar assim')
    salvar_tabela(R['dsr'], 'sharpe_deflacionado', indice=False)


def secao_capital_minimo():
    R['capital_minimo'] = capital_minimo(PESOS_E)
    print(resumo_capital(R['capital_minimo']).astype(float).round(0).to_string())
    print('piso para montar a carteira, em R$: não muda retorno nem carteira; a capacidade é o teto')
    salvar_tabela(R['capital_minimo'], 'capital_minimo')
    R['aporte_inicial'] = aporte_inicial(R['capital_minimo'], RET['iMOM-BR'])
    print('aporte único no primeiro mês que sustenta a carteira em todos os meses da fatia, sem novos aportes (bruto de custo e imposto, R$):')
    print(R['aporte_inicial'].to_string())
    salvar_tabela(R['aporte_inicial'], 'aporte_inicial')


def secao_capacidade():
    R['capacidade'] = {d: capacidade(PESOS_E, d) for d in DIAS_EXECUCAO}
    for d, c in R['capacidade'].items():
        v = c['capacidade_milhoes'].astype(float)
        print(f'{d} dia(s) de execução, participação {PARTICIPACAO:.0%}: mediana R$ {v.median():,.0f} mi | mínimo R$ {v.min():,.0f} mi em {v.idxmin().date()} ({c.loc[v.idxmin(), "limitante"]})')
    print('ações que mais limitam a capacidade:', R['capacidade'][1]['limitante'].value_counts().head(5).to_dict())
    salvar_tabela(pd.concat({f'{d}_dias': c for d, c in R['capacidade'].items()}, axis=1), 'capacidade')

## 6.6 Tabelas principais
Tabela só do iMOM-BR (retorno, Sharpe, volatilidade, queda máxima, beta e correlação com o Ibovespa etc., em %), tabela de comparação com as outras séries, correlação com o mercado e os fatores, as três maiores quedas, regime de juros e contribuição por ação nos meses extremos.

In [ ]:
def episodios_de_queda(r, topo=3):
    riqueza = (1 + r).cumprod()
    dd = riqueza / riqueza.cummax() - 1
    episodios, atual, ultimo_pico = [], None, 'início da fatia'
    for t in r.index:
        if dd[t] < 0:
            if atual is None:
                atual = {'pico': ultimo_pico, 'fundo': t.date(), 'queda_%': dd[t] * 100, 'recuperacao': 'não recuperou'}
            elif dd[t] * 100 < atual['queda_%']:
                atual['fundo'], atual['queda_%'] = t.date(), dd[t] * 100
        else:
            if atual is not None:
                atual['recuperacao'] = t.date()
                episodios.append(atual)
                atual = None
            ultimo_pico = t.date()
    if atual is not None:
        episodios.append(atual)
    return pd.DataFrame(episodios).sort_values('queda_%').head(topo).round({'queda_%': 2}).reset_index(drop=True)


def secao_tabelas_principais():
    F = 'iMOM-BR'
    r = R['resumo'].astype(float)
    R['comparacao'] = pd.DataFrame({'retorno acumulado (%)': r['retorno_acumulado'] * 100, 'retorno anualizado (%)': r['retorno_anual'] * 100,
                                    'volatilidade anual (%)': r['volatilidade_anual'] * 100, 'Sharpe': r['sharpe'], 'Sortino': r['sortino'],
                                    'queda máxima (%)': r['queda_maxima'] * 100, 'information ratio': r['information_ratio'],
                                    'tracking error (%)': r['tracking_error'] * 100}).round(2)
    print(R['comparacao'].to_string())
    salvar_tabela(R['comparacao'], 'comparacao')

    f = r.loc[F]
    beta = RET[F].cov(RET['Ibovespa']) / RET['Ibovespa'].var()
    R['estrategia'] = pd.Series({
        'meses': f['meses'], 'retorno acumulado (%)': f['retorno_acumulado'] * 100, 'retorno anualizado (%)': f['retorno_anual'] * 100,
        'retorno anualizado real, descontado o IPCA (%)': f['retorno_anual_real'] * 100, 'volatilidade anual (%)': f['volatilidade_anual'] * 100,
        'Sharpe': f['sharpe'], 'Sortino': f['sortino'], 'Calmar': f['calmar'], 'queda máxima (%)': f['queda_maxima'] * 100,
        'melhor mês (%)': f['melhor_mes'] * 100, 'pior mês (%)': f['pior_mes'] * 100, 'meses positivos (%)': f['meses_positivos'] * 100,
        'VaR 95% mensal (%)': f['VaR95_mensal'] * 100, 'CVaR 95% mensal (%)': f['CVaR95_mensal'] * 100, 'assimetria': f['assimetria'],
        'curtose em excesso': f['curtose_excesso'], 'beta contra o Ibovespa': beta, 'correlação com o Ibovespa': RET[F].corr(RET['Ibovespa']),
        'information ratio contra o Ibovespa': f['information_ratio'], 'tracking error (%)': f['tracking_error'] * 100,
        'captura de alta': f['captura_alta'], 'captura de baixa': f['captura_baixa'],
        'número médio de ações': (PESOS_E > 0).sum(axis=1).mean(), 'giro bilateral médio por mês': G[F].mean()}).astype(float).round(2).to_frame(FATIA)
    print(R['estrategia'].to_string())
    salvar_tabela(R['estrategia'], 'estrategia')

    fatores = dados['fatores_ff3'].reindex(MESES)
    R['correlacao'] = pd.concat([RET[[F, 'Residual sem filtro', 'Bruto com filtro', 'Bruto sem filtro', 'Só filtro', 'Ibovespa', 'Universo_igual']], fatores], axis=1).corr().round(2)
    print(R['correlacao'].to_string())
    salvar_tabela(R['correlacao'], 'correlacao')

    R['quedas'] = episodios_de_queda(RET[F])
    print(R['quedas'].to_string(index=False))
    salvar_tabela(R['quedas'], 'maiores_quedas', indice=False)

    delta_di = dados['di_mensal']['delta_taxa'].reindex(MESES)
    regime = pd.Series(np.where(delta_di < 0, 'DI em queda', 'DI em alta'), index=MESES)[delta_di.notna()]
    excesso = pd.DataFrame({c: RET.loc[regime.index, c] - CDI_A.loc[regime.index] for c in [F, 'Ibovespa', 'Universo_igual']})
    por_regime = excesso.groupby(regime).agg(['mean', 'count']).round(4)
    print(por_regime.to_string())
    print('diagnóstico descritivo: poucos ciclos e poucos meses, sem poder para teste')
    salvar_tabela(por_regime, 'regime_de_juros')

    aprovadas = pd.DataFrame({'com_sinal': sinal_residual.notna().sum(axis=1), 'aprovadas_com_sinal': (aprovada & sinal_residual.notna()).sum(axis=1),
                              'N_quintil': np.ceil(0.2 * sinal_residual.notna().sum(axis=1))}).loc[DECISAO]
    salvar_tabela(aprovadas, 'aprovadas_por_data')

    contrib = {}
    for s in MESES:
        cods = PESOS_E.columns[(PESOS_E.loc[s] > 0).to_numpy()]
        contrib[s] = PESOS_E.loc[s, cods] * retorno_no_mes(cods, s)
    contrib = pd.DataFrame(contrib).T
    k = min(3, len(MESES) // 2)
    extremos = RET[F].sort_values().iloc[list(range(k)) + list(range(-k, 0))]
    linhas = []
    for s, r_mes in extremos.items():
        c = contrib.loc[s].dropna().sort_values()
        for codigo in list(c.index[:3]) + list(c.index[-3:]):
            linhas.append({'mes': s.date(), 'retorno_iMOM': round(r_mes, 4), 'codigo': codigo, 'contribuicao': round(c[codigo], 4)})
    tabela_extremos = pd.DataFrame(linhas)
    print(tabela_extremos.to_string(index=False))
    salvar_tabela(tabela_extremos, 'contribuicao_meses_extremos', indice=False)

## 7.1 Padrão das figuras
Cores fixas por série (o iMOM-BR em azul-escuro e mais grossa) e a função que salva cada PNG como `04_<fatia>_NN_<nome>.png`.

In [ ]:
import matplotlib.dates as mdates

F = 'iMOM-BR'
from matplotlib.colors import LinearSegmentedColormap

ROXO_ESCURO, ROXO, ROXO_MEDIO, LILAS, LILAS_CLARO = '#230042', '#4B1A7A', '#6A3FA8', '#A88BD6', '#CDBDEB'
GRAFITE, CINZA, CINZA_CLARO = '#3A3742', '#6E6A78', '#A8A3B3'
CORES = {'iMOM-BR': ROXO_ESCURO, 'Residual sem filtro': ROXO_MEDIO, 'Bruto com filtro': LILAS, 'Bruto sem filtro': CINZA, 'Só filtro': CINZA_CLARO,
         'Ibovespa': CINZA, 'CDI': CINZA_CLARO, 'Universo_igual': LILAS}
TRACO = {'CDI': '--', 'Só filtro': '--'}
DIVERGENTE = LinearSegmentedColormap.from_list('cinza_roxo', [GRAFITE, '#BDB8C6', '#FFFFFF', LILAS, ROXO_ESCURO])
plt.rcParams.update({'figure.figsize': (9, 4.8), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
MESES_ABR = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
PASTA_ESTRATEGIA = PASTA_FIGURAS / 'estrategia'
PASTA_COMPARACOES = PASTA_FIGURAS / 'comparacoes'
PASTA_ESTRATEGIA.mkdir(parents=True, exist_ok=True)
PASTA_COMPARACOES.mkdir(parents=True, exist_ok=True)


def salvar(pasta, numero, nome):
    plt.tight_layout()
    plt.savefig(pasta / f'04_{FATIA}_{numero:02d}_{nome}.png', dpi=150)
    plt.show()
    plt.close('all')


def eixo_datas(eixo):
    eixo.set_xlim(MESES[0] - pd.offsets.MonthEnd(1), MESES[-1])
    eixo.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    eixo.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))


def acumulado_percentual(r):
    curva = ((1 + r).cumprod() - 1) * 100
    curva.loc[MESES[0] - pd.offsets.MonthEnd(1)] = 0.0
    return curva.sort_index()

## 7.2 Figuras do iMOM-BR
Retorno acumulado em %, queda desde o pico (com a maior queda marcada) e retorno mensal. Pasta `results/figuras/estrategia/`.

In [ ]:
def secao_figuras_estrategia():
    curva = acumulado_percentual(RET[F])
    fig, ax = plt.subplots()
    ax.plot(curva.index, curva, color=CORES[F], lw=2.4)
    ax.fill_between(curva.index, curva, 0, color=CORES[F], alpha=0.10)
    ax.axhline(0, color='k', lw=0.8)
    ax.annotate(f'{curva.iloc[-1]:.1f}%', (curva.index[-1], curva.iloc[-1]), textcoords='offset points', xytext=(0, -20), ha='center', fontsize=10, color=CORES[F])
    eixo_datas(ax)
    ax.set_title(f'iMOM-BR: retorno acumulado (%), bruto de custo ({FATIA})')
    ax.set_ylabel('% desde o início da fatia')
    salvar(PASTA_ESTRATEGIA, 1, 'retorno_acumulado')

    dd = curva_drawdown(RET[F]) * 100
    fig, ax = plt.subplots()
    ax.fill_between(dd.index, dd, 0, color=LILAS, alpha=0.35)
    ax.plot(dd.index, dd, color=ROXO_ESCURO, lw=1.8)
    fundo = dd.idxmin()
    ax.annotate(f'maior queda: {dd.min():.1f}% em {fundo:%m/%Y}', (fundo, dd.min()), textcoords='offset points', xytext=(0, -16), ha='center', fontsize=9)
    eixo_datas(ax)
    ax.set_title(f'iMOM-BR: queda desde o pico mais recente (%) ({FATIA})')
    ax.set_ylabel('%')
    salvar(PASTA_ESTRATEGIA, 2, 'queda_desde_o_pico')

    mapa = RET[F].to_frame('r')
    mapa['ano'], mapa['mes'] = mapa.index.year, mapa.index.month
    mapa = mapa.pivot(index='ano', columns='mes', values='r').reindex(columns=range(1, 13)) * 100
    fig, ax = plt.subplots(figsize=(9, 1.2 + 0.6 * len(mapa)))
    lim = np.nanmax(np.abs(mapa.to_numpy()))
    im = ax.imshow(mapa.to_numpy(), cmap=DIVERGENTE, vmin=-lim, vmax=lim, aspect='auto')
    ax.set_xticks(range(12), MESES_ABR)
    ax.set_yticks(range(len(mapa)), mapa.index)
    for i in range(mapa.shape[0]):
        for j in range(12):
            if not np.isnan(mapa.iat[i, j]):
                ax.text(j, i, f'{mapa.iat[i, j]:.1f}', ha='center', va='center', fontsize=8, color='white' if abs(mapa.iat[i, j]) > 0.6 * lim else 'black')
    ax.grid(False)
    ax.set_title(f'iMOM-BR: retorno mensal (%), bruto de custo ({FATIA})')
    fig.colorbar(im, ax=ax, label='%')
    salvar(PASTA_ESTRATEGIA, 3, 'retorno_mensal')

## 7.3 Carteira do iMOM-BR: setores e fatores
Setores que mais pesam na carteira contra o universo (e os 5 principais, impressos) e exposição do iMOM-BR a MKT, SMB e HML. Pasta `results/figuras/estrategia/`.

In [ ]:
def secao_figuras_carteira():
    setor = dados['cadastro_universo']['setor_economico'].replace('-', np.nan).fillna('Sem setor')
    universo = pd.concat([(elegivel.loc[t] & sinal_residual.loc[t].notna()).groupby(setor).sum() for t in DECISAO], axis=1).sum(axis=1)
    carteira = PESOS_E.T.groupby(setor).sum().sum(axis=1)
    comp = pd.DataFrame({'universo (elegíveis com sinal)': universo / universo.sum() * 100, 'carteira iMOM-BR': carteira / carteira.sum() * 100})
    comp = comp.fillna(0).sort_values('carteira iMOM-BR', ascending=False)
    print('principais setores da carteira do iMOM-BR (peso médio, %):')
    print(comp.head(5).round(1).to_string())
    salvar_tabela(comp.round(2), 'composicao_setorial')
    top = comp[comp['carteira iMOM-BR'] > 0.5].sort_values('carteira iMOM-BR')
    top.plot.barh(color=[CINZA_CLARO, CORES[F]], figsize=(9, 5.2))
    plt.title(f'iMOM-BR: setores da carteira contra o universo, peso médio (%) ({FATIA})')
    plt.ylabel('')
    salvar(PASTA_ESTRATEGIA, 4, 'composicao_setorial')

    b = R['fatores'].loc[F].astype(float)
    nomes = ['MKT', 'SMB', 'HML']
    y = [b['beta_MKT'], b['beta_SMB'], b['beta_HML']]
    erro = [[0, b['beta_SMB'] - b['SMB_ic_inf'], b['beta_HML'] - b['HML_ic_inf']], [0, b['SMB_ic_sup'] - b['beta_SMB'], b['HML_ic_sup'] - b['beta_HML']]]
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.bar(nomes, y, yerr=erro, color=CORES[F], capsize=4)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'iMOM-BR: exposição aos fatores (IC de 95% para SMB e HML) ({FATIA})', fontsize=9)
    salvar(PASTA_ESTRATEGIA, 5, 'exposicao_fatores')

## 7.4 Figuras de comparação
O iMOM-BR contra Ibovespa, universo igual e CDI; contra as outras versões; retorno por ano; intervalo da diferença de Sharpe; quintis do sinal e correlação com o mercado e os fatores. Pasta `results/figuras/comparacoes/`.

In [ ]:
def secao_figuras_comparacoes():
    curva = pd.concat({c: acumulado_percentual(RET[c]) for c in RET.columns}, axis=1)
    fig, ax = plt.subplots()
    for c in [F, 'Ibovespa', 'Universo_igual', 'CDI']:
        ax.plot(curva.index, curva[c], label=c.replace('Universo_igual', 'Universo (peso igual)'), color=CORES[c], lw=2.4 if c == F else 1.4, ls=TRACO.get(c, '-'))
    eixo_datas(ax)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'iMOM-BR contra as referências: retorno acumulado (%), bruto de custo ({FATIA})')
    ax.set_ylabel('% desde o início da fatia')
    ax.legend(frameon=False, ncol=4)
    salvar(PASTA_COMPARACOES, 1, 'curva_contra_referencias')

    fig, ax = plt.subplots()
    for c in [F, 'Residual sem filtro', 'Bruto sem filtro']:
        ax.plot(curva.index, curva[c], label=c, color=CORES[c], lw=2.4 if c == F else 1.5, ls=TRACO.get(c, '-'))
    eixo_datas(ax)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'iMOM-BR contra as outras versões: retorno acumulado (%) ({FATIA})')
    ax.set_ylabel('% desde o início da fatia')
    ax.legend(frameon=False, ncol=3)
    salvar(PASTA_COMPARACOES, 2, 'curva_contra_versoes')

    (R['anual'][[F, 'Ibovespa', 'CDI']] * 100).plot.bar(color=[CORES[F], CORES['Ibovespa'], CORES['CDI']], figsize=(9, 4.5))
    plt.title(f'Retorno por ano civil (%), meses da fatia {FATIA}')
    plt.xticks(rotation=0)
    salvar(PASTA_COMPARACOES, 3, 'retorno_por_ano')

    b = R['bootstrap'].drop(['iMOM-BR contra Só filtro', 'iMOM-BR contra Bruto com filtro'], errors='ignore')
    y = np.arange(len(b))[::-1]
    plt.hlines(y, b['limite_inferior'].astype(float), b['limite_superior'].astype(float), color=CORES[F], lw=2.5)
    plt.plot(b['diferenca_observada'].astype(float), y, 'o', color='k')
    plt.axvline(0, color=CINZA, lw=1)
    plt.yticks(y, list(b.index))
    plt.title(f'Diferença de Sharpe anualizado e IC de 95% ({FATIA})\n(bootstrap em blocos de 6 meses)', fontsize=10)
    plt.xlabel('iMOM-BR menos o comparador')
    salvar(PASTA_COMPARACOES, 4, 'intervalo_diferenca_sharpe')

    fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
    for eixo, amostra in zip(ax, ['elegíveis com sinal', 'aprovadas']):
        medias = R['quintis'][(amostra, 'residual')][[f'Q{i}' for i in range(1, 6)]].mean() * 100
        eixo.bar(medias.index, medias.to_numpy(), color=[LILAS_CLARO, LILAS, ROXO_MEDIO, ROXO, ROXO_ESCURO])
        eixo.axhline(0, color='k', lw=0.8)
        eixo.set_title(f'Retorno mensal médio por quintil do sinal residual (%)\n({amostra}, {FATIA})', fontsize=9)
    salvar(PASTA_COMPARACOES, 5, 'quintis_do_sinal')

    corr = R['correlacao']
    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    im = ax.imshow(corr.to_numpy(), cmap=DIVERGENTE, vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr)), corr.columns, rotation=40, ha='right')
    ax.set_yticks(range(len(corr)), corr.index)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f'{corr.iat[i, j]:.2f}', ha='center', va='center', fontsize=8, color='white' if abs(corr.iat[i, j]) > 0.6 else 'black')
    ax.grid(False)
    ax.set_title(f'Correlação dos retornos mensais, com o mercado e os fatores ({FATIA})')
    fig.colorbar(im, ax=ax)
    salvar(PASTA_COMPARACOES, 6, 'correlacao')

## 7.5 Rotina da fatia
`rodar_fatia` roda tudo de uma fatia; `rodar_se_liberada` só roda se ela estiver em `LIBERADAS`.

In [ ]:
def rodar_fatia(nome):
    definir_fatia(nome)
    secao_resumo()
    secao_custos()
    secao_bootstrap()
    secao_alfa()
    secao_sinal()
    secao_anos()
    secao_deflacionado()
    secao_capacidade()
    secao_capital_minimo()
    secao_tabelas_principais()
    if nome in FIGURAS_FATIAS:
        secao_figuras_estrategia()
        secao_figuras_carteira()
        secao_figuras_comparacoes()
    pd.DataFrame([t for t in registro_testes if t['fatia'] == nome]).to_csv(PASTA_TABELAS / f'04_{nome}_registro_de_testes.csv', index=False)
    print(f'fatia {nome}: {sum(t["fatia"] == nome for t in registro_testes)} testes registrados')


def rodar_se_liberada(nome):
    if nome in LIBERADAS:
        rodar_fatia(nome)
    else:
        print(f'fatia {nome} bloqueada: registrar a Emenda 6 e incluir "{nome}" em LIBERADAS antes de rodar')

## 8. Treino
Roda todas as seções com os meses da fatia de treino (16 meses, 09/2020 a 12/2021). **Serve só para provar que a mecânica roda; o Sharpe que importa é o da validação e o do teste.**

In [ ]:
rodar_se_liberada('treino')

## 9. Validação
Roda depois do treino, na mesma ordem do notebook. Na versão entregue `'validacao'` já está em `LIBERADAS`; antes disso (hoje) ela só imprime que está bloqueada.

In [ ]:
rodar_se_liberada('validacao')

## 10. Teste
Última fatia, lida uma única vez, com tudo congelado. Na versão entregue `'teste'` já está em `LIBERADAS` e roda depois da validação.

In [ ]:
rodar_se_liberada('teste')

## 11. Fatias lado a lado
Métricas principais de cada fatia liberada na mesma tabela e no mesmo gráfico.

In [ ]:
def lado_a_lado():
    linhas, estrategia = {}, {}
    for nome in [n for n in FATIAS if n in LIBERADAS]:
        meses = meses_da_fatia(nome)
        for c in ['iMOM-BR', 'Residual sem filtro', 'Bruto com filtro', 'Bruto sem filtro', 'Só filtro', 'Ibovespa', 'Universo_igual']:
            r = RET_TODO.loc[meses, c]
            linhas[(nome, c)] = {'meses': len(r), 'retorno_acumulado_%': acumulado(r) * 100, 'retorno_anual_%': cagr(r) * 100, 'volatilidade_anual_%': volatilidade(r) * 100,
                                 'sharpe': sharpe(r, cdi.loc[meses]), 'queda_maxima_%': curva_drawdown(r).min() * 100}
        m = resumo(RET_TODO.loc[meses, 'iMOM-BR'], cdi.loc[meses], ibov.loc[meses], ipca.loc[meses])
        estrategia[nome] = {'retorno acumulado (%)': m['retorno_acumulado'] * 100, 'retorno anualizado (%)': m['retorno_anual'] * 100, 'volatilidade anual (%)': m['volatilidade_anual'] * 100,
                            'Sharpe': m['sharpe'], 'Sortino': m['sortino'], 'queda máxima (%)': m['queda_maxima'] * 100, 'information ratio contra o Ibovespa': m['information_ratio']}
    tabela = pd.DataFrame(linhas).T.astype(float).round(2)
    print(tabela.to_string())
    tabela.to_csv(PASTA_TABELAS / '04_lado_a_lado.csv')
    por_fatia = pd.DataFrame(estrategia).astype(float).round(2)
    print(por_fatia.to_string())
    por_fatia.to_csv(PASTA_TABELAS / '04_estrategia_por_fatia.csv')
    anual = pd.DataFrame({n: {c: cagr(RET_TODO.loc[meses_da_fatia(n), c]) * 100 for c in ['iMOM-BR', 'Ibovespa', 'CDI']} for n in FATIAS if n in LIBERADAS}).T
    anual.plot.bar(color=[CORES['iMOM-BR'], CORES['Ibovespa'], CORES['CDI']])
    plt.title('Retorno anualizado por fatia (%)')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(PASTA_COMPARACOES / '04_lado_a_lado.png', dpi=150)
    plt.show()
    plt.close('all')


lado_a_lado()

## 11.1 Capital mínimo no período inteiro
Quanto é preciso ter para montar a carteira do iMOM-BR em cada mês, nas fatias liberadas: o primeiro mês, a média e os extremos. É um piso de implementação, não entra em retorno nem na escolha das ações. Com esse valor, a ação mais cara fica com 1 ação (ou 1 lote), e as outras são arredondadas para perto do peso igual.

**Aporte inicial que sustenta a estratégia.** Quanto aplicar uma única vez, no primeiro mês, para conseguir montar a carteira em todos os meses seguintes sem novos aportes. Não é a média do capital mínimo: em cada mês o capital exigido é dividido pelo quanto R$ 1 aplicado no início já rendeu até ali, e o aporte é o maior desses valores. É bruto de custo e imposto, então é um piso.

In [ ]:
def capital_do_periodo():
    meses = [m for nome in FATIAS if nome in LIBERADAS for m in meses_da_fatia(nome)]
    tabela = capital_minimo(bt['pesos']['E_residual_filtro'].loc[meses])
    resumo_geral = resumo_capital(tabela).astype(float).round(2)
    print(resumo_geral.to_string())
    print('ações que mais limitam o capital:', tabela['limitante'].value_counts().head(5).to_dict())
    tabela.to_csv(PASTA_TABELAS / '04_capital_minimo_periodo.csv')
    resumo_geral.to_csv(PASTA_TABELAS / '04_capital_minimo_resumo.csv')
    inicial = aporte_inicial(tabela, RET_TODO.loc[meses, 'iMOM-BR'])
    print('aporte único no primeiro mês que sustenta a carteira no período inteiro (bruto de custo e imposto, R$):')
    print(inicial.to_string())
    inicial.to_csv(PASTA_TABELAS / '04_aporte_inicial_periodo.csv')
    v = tabela['capital_1_acao'].astype(float)
    fig, ax = plt.subplots()
    ax.plot(v.index, v, color=CORES[F], lw=2)
    ax.axhline(v.mean(), color=CORES['Ibovespa'], lw=1, ls='--')
    ax.annotate(f'média: R$ {v.mean():,.0f}'.replace(',', '.'), (v.index[len(v) * 3 // 5], v.mean()), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=9)
    ax.annotate(f'primeiro mês: R$ {v.iloc[0]:,.0f}'.replace(',', '.'), (v.index[0], v.iloc[0]), textcoords='offset points', xytext=(0, 42), fontsize=9)
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
    ax.set_ylim(bottom=0)
    ax.set_title('iMOM-BR: capital mínimo para montar a carteira com 1 ação da mais cara (R$)')
    ax.set_ylabel('R$')
    plt.tight_layout()
    plt.savefig(PASTA_ESTRATEGIA / '04_capital_minimo.png', dpi=150)
    plt.show()
    plt.close('all')


capital_do_periodo()

## 12. Registro de testes
Lista de toda comparação rodada, por fatia, para a contagem do paper. Confere que a regra de bloqueio funciona para as fatias que ainda não foram liberadas.

In [ ]:
arquivos = sorted(PASTA_TABELAS.glob('04_*_registro_de_testes.csv'))
registro = pd.concat([pd.read_csv(a) for a in arquivos], ignore_index=True)
registro.to_csv(PASTA_TABELAS / '04_registro_de_testes.csv', index=False)
print(len(registro), 'testes registrados, por fatia:', registro['fatia'].value_counts().to_dict())
print('fatias liberadas:', LIBERADAS, '| ainda bloqueadas:', [n for n in FATIAS if n not in LIBERADAS] or 'nenhuma')
for nome in FATIAS:
    if nome not in LIBERADAS:
        try:
            meses_da_fatia(nome)
        except PermissionError as erro:
            print('bloqueio funciona:', erro)

## Resumo 
**O que o 04 entrega:** por fatia, 20 tabelas em `results/tabelas` (19 no treino, que não gera a composição setorial); figuras da validação e do teste em `results/figuras/estrategia` (só o iMOM-BR) e `results/figuras/comparacoes`; mais o lado a lado, o capital mínimo do período inteiro e o registro de testes.

**Antes de liberar a validação:** Emenda 6 registrada no PROTOCOLO.md com histórico mínimo, janela do desvio, base do N, alternativa sem candidatas, execução e cortes. **O teste é lido uma vez.**

**Limitações:** nos meses em que aprovadas ≤ N, o iMOM-BR compra todas as aprovadas e fica igual ao Só filtro e ao Bruto com filtro, e o resultado mede o filtro, não o sinal; com poucos meses os intervalos de bootstrap e de Newey-West são largos e pouco confiáveis; o Sharpe deflacionado usa a variância das variantes do 03, não das 315 configurações da exploração; o Ibovespa e o universo igual não pagam custo; resultados antes do imposto de renda.